# Import

In [1]:
from src.utils_worldmove import process_region, load_trajectory_lines
from src.utils_edit_graph import (handle_click,create_bridge)
from src.utils_elevation import download_opentopography_dem
from src.utils_computecost import bikespeed_from_power

import osmnx as ox
import networkx as nx
import geopandas as gpd
import math


from ipyleaflet import (
    Map,
    GeoJSON,
    Marker,
    Polyline,
    basemaps
)


# Falls nötig vorhher installieren

import geopandas as gpd
import osmnx as ox
import pyproj
import json

from shapely.geometry import LineString


print(f'OSMnx Verion: {ox.__version__}')
print(f'Networkx Verion: {nx.__version__}')

OSMnx Verion: 2.1.0
Networkx Verion: 3.6.1


# 1 Daten einlesen
[https://fi.ee.tsinghua.edu.cn/worldmove/](https://fi.ee.tsinghua.edu.cn/worldmove/)


In [2]:
# Process Administrative Boundaries
region, polygon, polygon_buffered = process_region(
    "data/Admin_Boundaries/regions.shp",  
    "output/City_Polygon.json", 
    target_crs="EPSG:4326",
    buffer_distance=1000)

# polygon.explore()

# -----------------------------------------------------------

# Process Trajectories

gdf_points, gdf_lines = load_trajectory_lines(
    "data/trajectories.npz",
    polygon=polygon,
    randomise=True,
    crs_output=4326
)
# gdf_points.head(3000).explore()

Regions reprojected to EPSG:4326
Prozess abgeschlossen.
[<POINT (8.51 47.4)> <POINT (8.51 47.4)> <POINT (8.52 47.4)> ...
 <POINT (8.58 47.3)> <POINT (8.58 47.3)> <POINT (8.58 47.3)>]


c:\Users\celin\Documents\00_FHNW\__Lokal-GIT\4040-Hackathon\BuildABridge\Final\src\utils_worldmove.py:192: UserWarning: Geometry is in a geographic CRS. Results from 'length' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf_lines["len"] = gdf_lines.length


## Menge der Reisen Reduzieren

In [3]:
samplesize = 3000 
gdf_lines.sample(n=samplesize)

,traj_idx,start_x,start_y,end_x,end_y,geometry,len
107515,3335,8.536714,47.379528,8.521746,47.380797,"LINESTRING (8.5367 47.38, 8.5217 47.381)",0.015021
245822,7619,8.509149,47.380582,8.507871,47.378973,"LINESTRING (8.5091 47.381, 8.5079 47.379)",0.002055
527768,16394,8.491505,47.408066,8.505085,47.401597,"LINESTRING (8.4915 47.408, 8.5051 47.402)",0.015042
236108,7313,8.532263,47.337647,8.530840,47.361008,"LINESTRING (8.5323 47.338, 8.5308 47.361)",0.023404
35825,1102,8.529949,47.363348,8.521257,47.353895,"LINESTRING (8.5299 47.363, 8.5213 47.354)",0.012841
...,...,...,...,...,...,...,...
653231,20273,8.515917,47.368428,8.520978,47.371815,"LINESTRING (8.5159 47.368, 8.521 47.372)",0.006090
415105,12894,8.538004,47.380628,8.528461,47.384511,"LINESTRING (8.538 47.381, 8.5285 47.385)",0.010303
473977,14749,8.479129,47.395453,8.480744,47.406797,"LINESTRING (8.4791 47.395, 8.4807 47.407)",0.011458
526427,16352,8.543585,47.430412,8.549458,47.439109,"LINESTRING (8.5436 47.43, 8.5495 47.439)",0.010493


# 2 Graph herunterladen

In [4]:
gdf_peri = polygon.to_crs(epsg=4326)
bbox = gdf_peri.union_all().envelope
bbox_gdf = gpd.GeoDataFrame(geometry=[bbox], crs=gdf_peri.crs)
bbox = bbox_gdf.geometry.iloc[0]

G_4326 = ox.graph.graph_from_polygon(bbox, network_type="drive", simplify=True)
G_3857 = ox.project_graph(G_4326)


# 3 Graph bearbeiten

In [5]:
# als gpdf speichern (pro EPSG)
nodes_4326, edges_4326 = ox.graph_to_gdfs(G_4326)
nodes_3857, edges_3857 = ox.graph_to_gdfs(G_3857)

# leere Listen für def handle click und create bridge definieren
clicked_lines = []
clicked_points = []
clicked_markers = []
created_edges = []

# Karte Darstellen mit Jupyter Leaflet (Leaflet in Lat/Lon, 4326 ! Routing in WebMerc 3857 da Metrisch!)
# nur mal Grunddefinition, wird unten mit Funktionen ergänzt

center_lat = nodes_4326.geometry.y.mean()
center_lon = nodes_4326.geometry.x.mean()

m = Map(
    center=(center_lat, center_lon),
    zoom=13,
    basemap=basemaps.OpenStreetMap.Mapnik
)

edges_json = json.loads(edges_4326.to_json())
nodes_json = json.loads(nodes_4326.to_json())

hover_style = {
    "color": "yellow",
    "fillColor": "yellow",
    "fillOpacity": 1.0
}

edge_layer = GeoJSON(
    data=edges_json,
    style={
        "color": "gray",
        "weight": 1
    }
)

node_layer = GeoJSON(
    data=nodes_json,
    point_style={
        "radius": 3,
        "color": "blue",
        "fillColor": "blue",
        "fillOpacity": 0.8,
        "weight": 1
    },
    hover_style=hover_style
)


m.add_layer(edge_layer)
m.add_layer(node_layer)

#m      # aktivieren um Karte bereits hier anzuzeigen

In [6]:
def create_bridge(click1, click2):

    # Klicks von WGS84 -> projiziertes CRS (WebMerc 3857) transformieren

    transformer = pyproj.Transformer.from_crs(
        "EPSG:4326",
        G_3857.graph["crs"],
        always_xy=True
    )

    x1, y1 = transformer.transform(click1[1], click1[0])
    x2, y2 = transformer.transform(click2[1], click2[0])

    # Nearest Nodes im projizierten Graphen

    node1 = ox.distance.nearest_nodes(
        G_3857,
        x1,
        y1
    )

    node2 = ox.distance.nearest_nodes(
        G_3857,
        x2,
        y2
    )

    # Koordinaten holen (projiziert -> Meter) für Längenberechung (Kosten)

    p1_3857 = (
        G_3857.nodes[node1]["x"],
        G_3857.nodes[node1]["y"]
    )

    p2_3857 = (
        G_3857.nodes[node2]["x"],
        G_3857.nodes[node2]["y"]
    )

    # Länge korrekt berechnen

    line_3857 = LineString([p1_3857, p2_3857])

    length = line_3857.length

    # Zurücktransformieren nach WGS84
    #    (für Darstellung auf Leaflet)

    reverse_transformer = pyproj.Transformer.from_crs(
        G_3857.graph["crs"],
        "EPSG:4326",
        always_xy=True
    )

    lon1, lat1 = reverse_transformer.transform(*p1_3857)
    lon2, lat2 = reverse_transformer.transform(*p2_3857)

    p1_4326 = (lon1, lat1)
    p2_4326 = (lon2, lat2)

    line_4326 = LineString([p1_4326, p2_4326])

    # Edge im projizierten Graphen hinzufügen

    G_3857.add_edge(
        node1,
        node2,
        length=float(length),
        geometry=line_3857,
        custom_edge=True
    )

    created_edges.append((G_3857, node1, node2))

    G_3857.add_edge(
        node2,
        node1,
        length=float(length),
        geometry=LineString([p2_3857, p1_3857]),
        custom_edge=True
    )

    created_edges.append((G_3857, node2, node1))


    # Edge zusätzlich im 4326 Graphen
    #    für Kartenanzeige

    G_4326.add_edge(
        node1,
        node2,
        length=float(length),
        geometry=line_4326,
        custom_edge=True
    )

    created_edges.append((G_4326, node1, node2))

    G_4326.add_edge(
        node2,
        node1,
        length=float(length),
        geometry=LineString([p2_4326, p1_4326]),
        custom_edge=True
    )

    created_edges.append((G_4326, node2, node1))

    # Linie auf Karte anzeigen  (! Leaflet erwartet Lat, Lon -> gpdf arbeitet mit Lon, Lat!)

    bridge_line = Polyline(
        locations=[
        (lat1, lon1),
        (lat2, lon2)
        ],
        color="red",
        weight=5
    )

    clicked_lines.append(bridge_line)

    m.add_layer(bridge_line)

In [7]:
def handle_click(**kwargs):     #kwargs = key-word-arguments

    # Nur echte Klicks verarbeiten
    if kwargs.get("type") != "click":
        return

    # Alte Marker/Linien/Edges entfernen
    if len(clicked_points) == 0:

        for marker in clicked_markers:
            m.remove_layer(marker)

        clicked_markers.clear()

        for line in clicked_lines:
            m.remove_layer(line)

        clicked_lines.clear()

        for graph, u, v in created_edges:

            if graph.has_edge(u, v):
                graph.remove_edge(u, v)

        created_edges.clear()

    # Klickkoordinaten
    lat, lon = kwargs.get("coordinates")

    # Punkt speichern, Marker sezten
    clicked_points.append((lat, lon))

    marker = Marker(location=(lat, lon))

    clicked_markers.append(marker)

    m.add_layer(marker)

    # Sobald zwei Punkte vorhanden, funktion von oben aufrufen
    if len(clicked_points) == 2:

        create_bridge(
            clicked_points[0],
            clicked_points[1]
        )

        # Punkte zurücksetzen
        clicked_points.clear()


In [8]:
# click Handler auf Interaktion (click) zu Karte hinzufügen
m.on_interaction(handle_click)

In [9]:
# karte Anzeigen und Funktionen Ausführen (Brücken Bauen ;)
# Jeweils eine Brücke aufs Mal, auch mehrer möglich -> Funktion handle click anpassen

m

Map(center=[47.36803176581384, 8.509037556306248], controls=(ZoomControl(options=['position', 'zoom_in_text', …

In [ ]:
G = G_4326

# Höhendaten hinzufügen

In [ ]:
import os

api_key = os.getenv("ELEVATION_API_KEY")

path = download_opentopography_dem(
    api_key=api_key,
    output_file="data/global_copernicus_90m.tif",
    demtype="COP90",
    south=47.244256757894696,
    north=47.47496700604382,
    west=8.345124410649325,
    east=8.637146966294877,
    output_format="GTiff",
)

print(f"Gespeichert unter: {path}")

In [ ]:
G = ox.elevation.add_node_elevations_raster(G, 'data/global_copernicus_90m.tif', band=1, cpus=None)
G = ox.elevation.add_edge_grades(G, add_absolute=True)

# Kosten Vergeben

In [ ]:
G = ox.projection.project_graph(G, to_crs=3857, to_latlong=False)
nodes, edges = ox.convert.graph_to_gdfs(G)


edges['bike_speed_mps'] = round(edges['grade'].apply(bikespeed_from_power),3)
edges['bike_traveltime'] = round(edges['length'] / edges['bike_speed_mps'],0)

nx.set_edge_attributes(G, edges['bike_speed_mps'].to_dict(), 'bike_speed_mps')
nx.set_edge_attributes(G, edges['bike_traveltime'].to_dict(), 'bike_traveltime')

# Start/Zielnodes vorbereiten

In [ ]:
gdf = gpd.read_file("trajectories_lines_l.json")

gdf["start_x"] = gdf.geometry.apply(
    lambda line: line.coords[0][0]
)

gdf["start_y"] = gdf.geometry.apply(
    lambda line: line.coords[0][1]
)

gdf["end_x"] = gdf.geometry.apply(
    lambda line: line.coords[-1][0]
)

gdf["end_y"] = gdf.geometry.apply(
    lambda line: line.coords[-1][1]
)
gdf["source"] = ox.distance.nearest_nodes(
    G,
    X=gdf["start_x"].values,
    Y=gdf["start_y"].values
)

gdf["target"] = ox.distance.nearest_nodes(
    G,
    X=gdf["end_x"].values,
    Y=gdf["end_y"].values
)

DataSourceError: trajectories_lines_l.json: No such file or directory

# Heuristic Skale

In [ ]:
def heuristic_scale_from_bbox(bbox, safety_factor=0.99):
    if hasattr(bbox, "centroid"):
        center_lat = bbox.centroid.y
    else:
        minx, miny, maxx, maxy = bbox
        center_lat = (miny + maxy) / 2

    return math.cos(math.radians(abs(center_lat))) * safety_factor

heuristic_scale = heuristic_scale_from_bbox(bbox)

print(bbox)
print(f"Heuristik-Skalierung fuer A*: {heuristic_scale:.3f}")